# Task 3 — Distributions, relationships, and Simpson’s paradox

**Week 1 data visualization assignment**

This notebook follows the task brief and chart-design rules: titles state findings, bars use a zero baseline, color carries meaning, excluded rows are stated, and each chart has a short rationale.

## Setup and data

Run the cells from top to bottom. The CSV is read from its public source URL; an internet connection is required. If needed, install `pandas`, `numpy`, and `matplotlib` in your notebook environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(DATA_URL)
df.head()

## Missing data audit and decision

Find missing measurements and sex values first. Keep rows with known body mass for body-mass plots; use complete bill pairs for the bill scatter; use rows with known sex for the sex comparison. Never impute these values for the requested descriptive comparisons. The chart captions will state the exclusions.

In [ ]:
print(df.isna().sum())
print('Rows missing all four measurements:', int(df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].isna().all(axis=1).sum()))
print('Rows missing sex:', int(df.sex.isna().sum()))
body = df.dropna(subset=['body_mass_g'])
bills = df.dropna(subset=['bill_length_mm','bill_depth_mm','species'])
sex_mass = df.dropna(subset=['body_mass_g','sex','species'])
print(f'Body-mass charts exclude {len(df)-len(body)} rows; bill scatter excludes {len(df)-len(bills)} rows; sex comparison excludes {len(df)-len(sex_mass)} rows.')

## Overall body-mass distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(body.body_mass_g, bins=22, color='#5B9BD5', edgecolor='white')
ax.set(title='Penguin body mass has multiple overlapping size groups', xlabel='Body mass (g)', ylabel='Number of penguins')
ax.text(.01,.98,f"Excludes {len(df)-len(body)} rows with missing body mass",transform=ax.transAxes,va='top',fontsize=9)
fig.tight_layout()

**Why a histogram:** body mass is continuous, and bins show its distribution and possible modes. The pooled distribution has more than one hump, suggesting distinct groups.

## Body-mass distribution by species

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
order = sorted(body.species.unique())
for i, species in enumerate(order):
    vals = body.loc[body.species.eq(species),'body_mass_g']
    ax.hist(vals, bins=18, alpha=.5, label=species, edgecolor='white')
ax.set(title='Species have different body-mass ranges, explaining the pooled humps', xlabel='Body mass (g)', ylabel='Number of penguins')
ax.legend(title='Species', frameon=False)
ax.text(.01,.98,f"Excludes {len(df)-len(body)} rows with missing body mass",transform=ax.transAxes,va='top',fontsize=9)
fig.tight_layout()

**Why split by species:** translucent histograms compare the groups and show how their different mass ranges combine into the overall shape. Species is the meaningful color grouping.

## Bill length and depth: pooled versus within-species relationships

In [ ]:
pooled_r = bills.bill_length_mm.corr(bills.bill_depth_mm)
by_species = {name: g.bill_length_mm.corr(g.bill_depth_mm) for name, g in bills.groupby('species')}
fig, ax = plt.subplots(figsize=(8,6))
colors = {'Adelie':'#0072B2','Chinstrap':'#D55E00','Gentoo':'#009E73'}
for species, g in bills.groupby('species'):
    ax.scatter(g.bill_length_mm,g.bill_depth_mm,s=32,alpha=.75,label=species,color=colors.get(species))
ax.set(title='Bill length and depth reverse direction when species are pooled',xlabel='Bill length (mm)',ylabel='Bill depth (mm)')
ax.legend(title='Species',frameon=False)
ax.text(.02,.02,f'Pooled r = {pooled_r:.2f}; within species: '+', '.join(f'{k} r={v:.2f}' for k,v in by_species.items())+f'\nExcludes {len(df)-len(bills)} rows missing either bill measurement',transform=ax.transAxes,fontsize=9)
fig.tight_layout()

**What is happening:** the pooled association is negative while the within-species associations are positive. This is Simpson’s paradox: species groups occupy different regions of the chart, and pooling them changes the direction of the association. The within-group pattern answers a different question from the pooled pattern, so group membership matters.

## Average body mass by species and sex

In [ ]:
means = sex_mass.groupby(['species','sex']).body_mass_g.mean().unstack()
ax = means.plot(kind='bar',figsize=(9,5),color=['#CC79A7','#56B4E9'])
ax.set(title='Average body mass differs by species and sex',xlabel='Species',ylabel='Mean body mass (g)')
ax.legend(title='Sex',frameon=False)
ax.text(.01,.98,f"Excludes {len(df)-len(sex_mass)} rows missing sex or body mass",transform=ax.transAxes,va='top',fontsize=9)
plt.xticks(rotation=0); plt.tight_layout()

**Why grouped bars:** bars compare mean mass across the species-by-sex combinations, with a zero baseline so the magnitude comparison is honest. The chart excludes rows with missing sex or body mass; the note reports the count.